In [1]:
import lancedb
from ranx import Qrels, Run, evaluate, compare
from lancedb.query import MatchQuery
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

In [2]:
import sys
sys.path.append("../src")
from rerank.rerankhf import rank


In [3]:
model_id = "cnmoro/Tucano160m-Reranker-EnPt"
model = AutoModelForSequenceClassification.from_pretrained(
        model_id,
        num_labels=2
    )
tokenizer = AutoTokenizer.from_pretrained(model_id)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
max_length = 2048

Loading weights:   0%|          | 0/111 [00:00<?, ?it/s]

In [4]:
db = lancedb.connect("../data/lancedb_mldr")


In [5]:
tb_passages = db.open_table('train_reranking_passages')
tb_qrels = db.open_table('train_reranking_qrels')
tb_queries = db.open_table('train_reranking_queries')


In [6]:
tb_passages.count_rows()

4692

In [7]:
qrels = tb_qrels.search().to_pandas()
qrels.sort_values(by="query_id", inplace=True)
len(qrels)


1683

In [8]:
def query_pandas_dict(df, query_col, doc_col, score_col):
    qid_atual = None
    q_docs = {}
    qrels_dict = {}
    for _, q in df.iterrows():
        if qid_atual is None:
            qid_atual = q[query_col]
            q_docs = {}

        if qid_atual != q[query_col]:
            qrels_dict[qid_atual] = q_docs
            qid_atual = q[query_col]
            q_docs = {}
        q_docs[q[doc_col]] = q[score_col]
    qrels_dict[qid_atual] = q_docs
    return qrels_dict

In [9]:
qrels_dict = query_pandas_dict(qrels, "query_id", "passage_id", "score")
qrels_dataset = Qrels(qrels_dict)
print(len(qrels_dict))


1683


In [10]:
print('q-pt-1049' in list(qrels_dict.keys()))
print('q-pt-222' in list(qrels_dict.keys()))

False
False


In [11]:
# disable_scoring_autoprojection
#tb_passages.search(query="Qual a receita de Fuji?", query_type="fts").select(["passage_id", "passage"]).limit(2).to_pandas()

df = (
    tb_passages.search(query="Qual a receita de Fuji?", query_type="fts")
    .limit(2)
    .to_pandas()[["passage_id", "passage", "_score"]]
)
df.shape
df

,passage_id,passage,_score
0,doc-pt-13571,Torre de Tóquio\n\nA é uma torre de comunicaçã...,11.220190
1,doc-pt-2602,Dragon Ball Z\n\n (geralmente abreviado como D...,9.202723


In [12]:
MAX_RESULTS = 10
def get_results_dict(qrels_dict):
    results_dict = {}
    query_documents = {}
    for k in qrels_dict.keys(): # varrer banco para cada query_id e pegar os resultados do modelo
        q = tb_queries.search().where(f"query_id = '{k}'").limit(1).to_pandas() # recupera a query do banco de dados
        s = tb_passages.search(MatchQuery(q["query"].iloc[0], "passage"), query_type="fts").limit(MAX_RESULTS).to_pandas()[["passage_id", "passage", "_score"]]
        if not s.empty:
            query_documents[k] = (q["query"].iloc[0], s["passage"].tolist(), s["passage_id"].tolist())
            s["query_id"] = k
            qrels_dict = query_pandas_dict(s, "query_id", "passage_id", "_score")
            try:
                results_dict[k] = qrels_dict[k]
            except KeyError as e:
                print(f"Query {k} not found in results_dict: {e}")
    return results_dict, query_documents

results_dict, query_documents = get_results_dict(qrels_dict)
print(len(results_dict))
print(len(query_documents))
# -------------------------------------------------------------
# Avaliação de um modelo de recuperação de informação (BM25) usando a biblioteca ranx
bm25_dataset = Run(results_dict, name="bm25")


1683
1683


In [13]:
# -------------------------------------------------------------
# Avaliação de um único sistema
# -------------------------------------------------------------
metrics = ["ndcg@10", "map@10", "mrr@10", "recall@10", "hit_rate@10"]
scores = evaluate(qrels_dataset, bm25_dataset, metrics)

print("Métricas Sistema Denso:")
for m, v in scores.items():
    print(f"  {m:<12}: {v:.4f}")


Métricas Sistema Denso:
  ndcg@10     : 0.8673
  map@10      : 0.8443
  mrr@10      : 0.8443
  recall@10   : 0.9382
  hit_rate@10 : 0.9382


In [14]:
#Métricas Sistema Denso (BM25 max 100):
#  ndcg@10     : 0.8673
#  map@10      : 0.8443
#  mrr@10      : 0.8443
#  recall@10   : 0.9382
#  hit_rate@10 : 0.9382

In [15]:
#query_documents["q-pt-1"]
ranked_results = rank(query_documents["q-pt-1"][0], query_documents["q-pt-1"][1], tokenizer, model, device, normalize_scores=True)


In [16]:
scores_docid = []
for i in range(len(ranked_results)):
    scores_docid.append((query_documents["q-pt-1"][2][i], ranked_results[i][1]))

sorted_scores_docid = sorted(scores_docid, key=lambda x: x[1], reverse=True)


In [17]:
result_reranked = { "q-pt-1": [x[0] for x in sorted_scores_docid[:10]] }
result_reranked

{'q-pt-1': ['doc-pt-3875',
  'doc-pt-6477',
  'doc-pt-8',
  'doc-pt-6510',
  'doc-pt-4961',
  'doc-pt-2793',
  'doc-pt-6039',
  'doc-pt-6953',
  'doc-pt-4285',
  'doc-pt-6853']}

In [ ]:
result_reranked = {}
for query_id, (query, documents, doc_ids) in query_documents.items():
    print(f"Reranking query: {query_id}")
    ranked_results = rank(query, documents, tokenizer, model, device, normalize_scores=True)
    scores_docid = []
    for i in range(len(ranked_results)):
        scores_docid.append((doc_ids[i], ranked_results[i][1]))
    sorted_scores_docid = sorted(scores_docid, key=lambda x: x[1], reverse=True)
    result_reranked[query_id] = {x[0]: x[1] for x in sorted_scores_docid}
reranked_results_dataset = Run(result_reranked, name=f"reranked_{model_id.split('/')[-1]}")


Reranking query: q-pt-1


In [ ]:
query_documents["q-pt-1049"]

KeyError: 'q-pt-1049'

In [ ]:
# -------------------------------------------------------------
# Comparação entre modelos com teste estatístico
# -------------------------------------------------------------
report = compare(
    qrels=qrels_dataset,
    runs=[run_bm25, run_dense],
    metrics=["ndcg@10", "mrr@10", "recall@10"],
    stat_test="student_t"  # Realiza teste t pareado
)
print("\nRelatório Comparativo:")
print(report)

NameError: name 'run_bm25' is not defined